In [18]:
import pandas as pd
import warnings
from mlxtend.frequent_patterns import apriori, association_rules
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [19]:
# Carregar o dataset
df = pd.read_csv("basket_supermercado_1000.csv")
pd.set_option('display.max_columns', None)
df = df.drop(columns=["transacao"])
print(df.head())

   pao  leite  cafe  manteiga  acucar  arroz  feijao  macarrao  carne  frango  \
0    1      1     0         0       0      0       0         0      0       0   
1    1      1     1         0       1      1       1         0      1       0   
2    1      1     0         0       0      1       0         0      0       1   
3    1      1     0         0       0      0       0         0      0       0   
4    0      0     0         0       0      1       1         0      1       1   

   peixe  ovos  queijo  presunto  cerveja  refrigerante  vinho  hortifruti  \
0      0     0       0         0        1             1      0           0   
1      0     0       0         0        0             0      0           0   
2      0     0       0         0        0             0      0           0   
3      0     0       0         0        1             1      0           0   
4      0     0       0         0        0             0      0           1   

   doces  limpeza  
0      1        0  
1   

In [20]:
# Aplicar o algoritmo Apriori
frequent_itemsets = apriori(
    df,
    min_support=0.05,
    use_colnames=True
)

print("Itemsets/Subconjuntos frequentes")
print(frequent_itemsets.sort_values(by="support", ascending=False))

Itemsets/Subconjuntos frequentes
     support                                       itemsets
0      0.633                                          (pao)
1      0.630                                        (leite)
14     0.624                                   (pao, leite)
5      0.519                                        (arroz)
6      0.512                                       (feijao)
..       ...                                            ...
260    0.050                  (limpeza, hortifruti, acucar)
573    0.050        (hortifruti, feijao, pao, acucar, cafe)
560    0.050      (hortifruti, feijao, leite, pao, cerveja)
601    0.050      (hortifruti, feijao, leite, acucar, cafe)
633    0.050  (hortifruti, arroz, leite, pao, acucar, cafe)

[662 rows x 2 columns]


In [21]:
# Gerar as regras de associação
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.6
)

print("Regras de associação")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
print(rules)

Regras de associação
                      antecedents                     consequents  \
0                           (pao)                         (leite)   
1                         (leite)                           (pao)   
2                          (cafe)                           (pao)   
3                      (manteiga)                           (pao)   
4                        (acucar)                           (pao)   
...                           ...                             ...   
2197  (pao, frango, acucar, cafe)          (arroz, leite, feijao)   
2198      (frango, acucar, leite)      (pao, arroz, cafe, feijao)   
2199        (frango, cafe, leite)    (pao, arroz, acucar, feijao)   
2200        (pao, frango, acucar)    (cafe, arroz, leite, feijao)   
2201          (pao, frango, cafe)  (acucar, arroz, leite, feijao)   

      antecedent support  consequent support  support  confidence      lift  \
0                  0.633               0.630    0.624    0.985782  1.56

In [22]:
# Filtrar as regras relevantes
rules_filtered = rules[
    (rules["lift"] > 1) &
    (rules["confidence"] >= 0.6)
].sort_values(by="lift", ascending=False)

print("Regras relevantes")

print(rules_filtered[["antecedents", "consequents", "support", "confidence", "lift"]])

Regras relevantes
                             antecedents                 consequents  support  \
1972       (refrigerante, leite, feijao)       (pao, arroz, cerveja)    0.059   
1973         (pao, refrigerante, feijao)     (arroz, leite, cerveja)    0.059   
1969        (refrigerante, arroz, leite)      (pao, feijao, cerveja)    0.059   
1970          (pao, arroz, refrigerante)    (leite, feijao, cerveja)    0.059   
1971            (leite, feijao, cerveja)  (pao, arroz, refrigerante)    0.059   
...                                  ...                         ...      ...   
175                (hortifruti, cerveja)                     (leite)    0.111   
523              (frango, arroz, feijao)                       (pao)    0.171   
112                     (frango, feijao)                       (pao)    0.171   
1512  (carne, hortifruti, arroz, feijao)                     (leite)    0.075   
690           (carne, hortifruti, arroz)                     (leite)    0.075   

      con

In [23]:
# Insights para o gerente
# Criar filtros
rules['antecedent_len'] = rules['antecedents'].apply(len)
rules['consequent_len'] = rules['consequents'].apply(len)

rules_simples = rules[
    (rules['antecedent_len'] <= 2) &
    (rules['consequent_len'] <= 2) &
    (rules['confidence'] > 0.6) &
    (rules['lift'] > 1.2)
]

top_rules = rules_simples.sort_values(by="lift", ascending=False).head(10)

for _, row in top_rules.iterrows():
    antecedente = ", ".join(list(row["antecedents"]))
    consequente = ", ".join(list(row["consequents"]))

    print(f"Se o cliente compra [{antecedente}], há {row['confidence']:.2f} de chance de também comprar [{consequente}]. (lift={row['lift']:.2f})")

Se o cliente compra [cafe, refrigerante], há 0.87 de chance de também comprar [acucar, cerveja]. (lift=6.61)
Se o cliente compra [acucar, refrigerante], há 0.86 de chance de também comprar [cafe, cerveja]. (lift=6.58)
Se o cliente compra [frango, refrigerante], há 0.94 de chance de também comprar [feijao, cerveja]. (lift=5.64)
Se o cliente compra [frango, refrigerante], há 0.94 de chance de também comprar [arroz, cerveja]. (lift=5.57)
Se o cliente compra [refrigerante, feijao], há 0.92 de chance de também comprar [arroz, cerveja]. (lift=5.50)
Se o cliente compra [arroz, refrigerante], há 0.91 de chance de também comprar [feijao, cerveja]. (lift=5.46)
Se o cliente compra [doces, feijao], há 0.87 de chance de também comprar [arroz, cerveja]. (lift=5.20)
Se o cliente compra [doces, arroz], há 0.85 de chance de também comprar [feijao, cerveja]. (lift=5.10)
Se o cliente compra [arroz, cafe], há 0.92 de chance de também comprar [acucar, feijao]. (lift=4.56)
Se o cliente compra [acucar, feija

In [24]:
print("Produtos mais comprados:")
print(df.sum().sort_values(ascending=False))

Produtos mais comprados:
pao             633
leite           630
arroz           519
feijao          512
hortifruti      490
cafe            393
acucar          390
manteiga        346
cerveja         338
frango          288
carne           261
limpeza         252
refrigerante    210
doces           131
macarrao         39
peixe            37
vinho            32
presunto         32
ovos             31
queijo           31
dtype: int64
